# Onboarding Agent

In [ ]:
import asyncio
from crewai import Agent, Crew, Task, Process
from crewai.tools import BaseTool
from pydantic import Field, BaseModel
from typing import Type
import re
import os

# Set up API key

os.environ['OPENAI_API_KEY'] = "YOUR_OPENAI_API_KEY"

### Custom Tool

In [2]:
class EmailGeneratorInput(BaseModel):
    fullname: str = Field(..., description="The full name of the employee.")
    department: str = Field(..., description="The department of the employee.")

# --- Custom Tool Definition ---
class EmailGeneratorTool(BaseTool):
    name: str = "Email Generator"
    description: str = "Generates a company email based on the employee's full name and department."
    args_schema: Type[BaseModel] = EmailGeneratorInput

    def _run(self, fullname: str, department: str) -> str:
            """Generate a company email from fullname and department."""
            name_part = re.sub(r'[^a-zA-Z]', '', fullname.replace(' ', '.')).lower()
            department_part = department.replace(' ', '').lower()
            return f"{name_part}@{department_part}.company.com"

### Agents

In [3]:
scheduler_agent = Agent(
    role="Scheduler",
    goal="Validate and confirm employee start date",
    backstory="Ensures the selected start date is valid and not a holiday."
)

it_agent = Agent(
    role="IT Administrator",
    goal="Set up company email and provide PC setup instructions",
    backstory="Prepares company email accounts and delivers simple PC setup guidelines.",
    tools=[EmailGeneratorTool()]
)

report_agent = Agent(
    role="Onboarding Reporter",
    goal="Generate an onboarding summary report",
    backstory="Creates a final onboarding report with all key details for HR records."
)

### Tasks

In [4]:
validate_start_date_task = Task(
    description=(
        "Check if the start date ({employe}) is a working day. "
        "If it's a holiday or weekend, suggest the next available working day."
    ),
    expected_output="Confirmed valid start date or suggestion for a new date.",
    agent=scheduler_agent
)

it_setup_task = Task(
    description=(
        "Create a company email for {employe} in the {employe} department. "
        "Provide a short, clear set of bullet points on how to set up their PC."
    ),

    expected_output="Active company email and clear PC setup instructions.",
    agent=it_agent
)

generate_report_task = Task(
    description=(
        "Generate a summary onboarding report for {employe}, starting on {employe}, "
        "in the {employe} department. Include email details, PC setup info, and start date confirmation."
    ),
    expected_output="A clear and concise onboarding summary report.",
    agent=report_agent
)


### Crew

In [5]:
# --- Crew Definition ---
onboarding_crew = Crew(
    agents=[scheduler_agent, it_agent, report_agent],
    tasks=[validate_start_date_task, it_setup_task, generate_report_task],
    process=Process.sequential,
    verbose=True
)

### Asyncronous Function

In [6]:
async def async_onboarding_execution(onboarding_inputs):
    results = await onboarding_crew.kickoff_async(inputs=onboarding_inputs)
    print("Crew result:", results.raw)

### Main

In [7]:
import nest_asyncio
nest_asyncio.apply()


onboarding_inputs = {
    "employe": [
        "Alice Johnson, starting 2024-07-15, department Engineering",
        "Bob Smith, starting 2024-07-16, department Marketing",
        "Charlie Brown, starting 2024-07-17, department Sales"
    ]
}

# Run asynchronous onboarding workflows
asyncio.run(async_onboarding_execution(onboarding_inputs))

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f063c0ed-c2ed-4e7c-b3dd-69863ca98063                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Check if the start date (['Alice Johnson, starting 2024-07-15, department Engineering', 'Bob Smith,      │
│  starting 2024-07-16, department Marketing', 'Charlie Brown, starting 2024-07-17, department Sales']) is a      │
│  working day. If it's a holiday or weekend, suggest the next available working day.                             │
│  ID: 6c057f7d-b0f0-4f68-b9c4-b76d2b3dcc8a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scheduler                                                                                               │
│                                                                                                                 │
│  Task: Check if the start date (['Alice Johnson, starting 2024-07-15, department Engineering', 'Bob Smith,      │
│  starting 2024-07-16, department Marketing', 'Charlie Brown, starting 2024-07-17, department Sales']) is a      │
│  working day. If it's a holiday or weekend, suggest the next available working day.                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Scheduler                                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Alice Johnson, starting 2024-07-15, department Engineering                                                     │
│  - 2024-07-15 is a Monday and not a holiday. Confirmed valid start date.                                        │
│                                                                                                                 │
│  Bob Smith, starting 2024-07-16, department Marketing                                                           │
│  - 2024-07-16 is a Tuesday and not a holiday. Confirmed valid start date.                                       │
│                                                                                                                 │
│  Charlie Brown, starting 2024-07-17, department Sales                                                           │
│  - 2024-07-17 is a Wednesday and not a holiday. Confirmed valid start date.                                     │
│                                                                                                                 │
│  All proposed start dates are valid working days and confirmed.                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Check if the start date (['Alice Johnson, starting 2024-07-15, department Engineering', 'Bob Smith,      │
│  starting 2024-07-16, department Marketing', 'Charlie Brown, starting 2024-07-17, department Sales']) is a      │
│  working day. If it's a holiday or weekend, suggest the next available working day.                             │
│  Agent: Scheduler                                                                                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Onboarding Reporter                                                                                     │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Onboarding Summary Report                                                                                      │
│                                                                                                                 │
│  1. Alice Johnson                                                                                               │
│  - Department: Engineering                                                                                      │
│  - Start Date: July 15, 2024 (Monday) – Confirmed valid working day                                             │
│  - Company Email: alicejohnson@engineering.company.com                                                          │
│  - PC Setup Instructions:                                                                                       │
│    • Power on your PC and complete the initial Windows setup.                                                   │
│    • Connect to the company Wi-Fi network using provided credentials.                                           │
│    • Log in with your company email and temporary password (sent separately).                                   │
│    • Install necessary software from the company portal (e.g., office suite, antivirus).                        │
│    • Configure your email client using your new company email address.                                          │
│    • Set up multi-factor authentication as instructed.                                                          │
│    • Save important documents to the company cloud storage.                                                     │
│    • Contact IT support if any issues arise.                                                                    │
│                                                                                                                 │
│  2. Bob Smith                                                                                                   │
│  - Department: Marketing                                                                                        │
│  - Start Date: July 16, 2024 (Tuesday) – Confirmed valid working day                                            │
│  - Company Email: bobsmith@marketing.company.com                                                                │
│  - PC Setup Instructions:                                                                                       │
│    • Power on your PC and complete the initial Windows setup.                                                   │
│    • Connect to the company Wi-Fi network using provided credentials.                                           │
│    • Log in with your company email and temporary password (sent separately).                                   │
│    • Install necessary software from the company portal (e.g., office suite, antivirus).                        │
│    • Configure your email client using your new company email address.                                          │
│    • Set up multi-factor authentication as instructed.                                                          │
│    • Save important documents to the company cloud storage.                                                     │
│    • Contact IT support if any issues arise.                                                                    │
│                                                        

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Generate a summary onboarding report for ['Alice Johnson, starting 2024-07-15, department Engineering',  │
│  'Bob Smith, starting 2024-07-16, department Marketing', 'Charlie Brown, starting 2024-07-17, department        │
│  Sales'], starting on ['Alice Johnson, starting 2024-07-15, department Engineering', 'Bob Smith, starting       │
│  2024-07-16, department Marketing', 'Charlie Brown, starting 2024-07-17, department Sales'], in the ['Alice     │
│  Johnson, starting 2024-07-15, department Engineering', 'Bob Smith, starting 2024-07-16, department             │
│  Marketing', 'Charlie Brown, starting 2024-07-17, department Sales'] department. Include email details, PC      │
│  setup info, and start date confirmation.                                                                       │
│  Agent: Onboarding Reporter                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Crew result: Onboarding Summary Report

1. Alice Johnson
- Department: Engineering
- Start Date: July 15, 2024 (Monday) – Confirmed valid working day
- Company Email: alicejohnson@engineering.company.com
- PC Setup Instructions:
  • Power on your PC and complete the initial Windows setup.
  • Connect to the company Wi-Fi network using provided credentials.
  • Log in with your company email and temporary password (sent separately).
  • Install necessary software from the company portal (e.g., office suite, antivirus).
  • Configure your email client using your new company email address.
  • Set up multi-factor authentication as instructed.
  • Save important documents to the company cloud storage.
  • Contact IT support if any issues arise.

2. Bob Smith
- Department: Marketing
- Start Date: July 16, 2024 (Tuesday) – Confirmed valid working day
- Company Email: bobsmith@marketing.company.com
- PC Setup Instructions:
  • Power on your PC and complete the initial Windows setup.
  • Conne

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f063c0ed-c2ed-4e7c-b3dd-69863ca98063                                                                       │
│  Final Output: Onboarding Summary Report                                                                        │
│                                                                                                                 │
│  1. Alice Johnson                                                                                               │
│  - Department: Engineering                                                                                      │
│  - Start Date: July 15, 2024 (Monday) – Confirmed valid working day                                             │
│  - Company Email: alicejohnson@engineering.company.com                                                          │
│  - PC Setup Instructions:                                                                                       │
│    • Power on your PC and complete the initial Windows setup.                                                   │
│    • Connect to the company Wi-Fi network using provided credentials.                                           │
│    • Log in with your company email and temporary password (sent separately).                                   │
│    • Install necessary software from the company portal (e.g., office suite, antivirus).                        │
│    • Configure your email client using your new company email address.                                          │
│    • Set up multi-factor authentication as instructed.                                                          │
│    • Save important documents to the company cloud storage.                                                     │
│    • Contact IT support if any issues arise.                                                                    │
│                                                                                                                 │
│  2. Bob Smith                                                                                                   │
│  - Department: Marketing                                                                                        │
│  - Start Date: July 16, 2024 (Tuesday) – Confirmed valid working day                                            │
│  - Company Email: bobsmith@marketing.company.com                                                                │
│  - PC Setup Instructions:                                                                                       │
│    • Power on your PC and complete the initial Windows setup.                                                   │
│    • Connect to the company Wi-Fi network using provided credentials.                                           │
│    • Log in with your company email and temporary password (sent separately).                                   │
│    • Install necessary software from the company portal (e.g., office suite, antivirus).                        │
│    • Configure your email client using your new company email address.                                          │
│    • Set up multi-factor authentication as instructed.                                                          │
│    • Save important documents to the company cloud storage.                                                     │
│    • Contact IT support if any issues arise.                                                                    │
│                                                       

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯